# Real visual attacks on VLMs: evidence and experiment laboratory
    **October 25, 2026 — implementation, not a completed study.**

    This notebook embeds real input optimizers, a differentiable Qwen2.5-VL adapter and
    a private paired-generation runner. Unlike the teaching notebook, it can load and attack
    a pretrained VLM when explicitly enabled with private data and an accepted compute scope.
    Default Run all is CPU acceptance checks. No paper reproduction or novel claim is established.
    All implemented paper-derived methods are adaptations unless independently verified otherwise.


In [1]:
# Leave these false for reading and offline acceptance checks.
RUN_GPU = False
INSTALL_REVIEWED_DEPENDENCIES = False
MOUNT_DRIVE = False
COMPUTE_APPROVED = False
SCIENTIFIC_AMENDMENT_ACCEPTED = False
APPROVED_SCOPE_SHA256 = ''
MANIFEST_PATH = ''  # Private JSON list. Never paste benchmark text into this notebook.
OUT_DIR = ''  # New private run folder under mounted /content/drive/MyDrive/.
METHOD = 'pgd'  # Also: fgsm/patch/universal/universal_patch/universal_eot/feature/prompt_match/injection/bimodal
DEFENSE_AWARE = False  # Require a fixed private defense_prefix; unsupported for feature/injection.
MAX_WALL_MINUTES = 30  # Starts BEFORE install/load; do not reset on resume.
MAX_NEW_TOKENS = 256
RUN_ID = 'visual_development_v1'


### Readout — editable

GPU is off. Keep earlier completed notebooks intact. Use a new dated run folder; save actual code and outputs plus the numeric status snapshot before changing methods.


## 1. What is already present — paper coverage

| Paper | Checked mathematical source | Entry point | Coverage |
|---|---|---|---|
| P01 FGSM | 1412.6572v3 §§3–4 | fgsm | Implemented generative adaptation; not original classifier reproduction |
| P02 PGD | 1706.06083v4 §2 | pgd | Implemented constrained input optimizer; no adversarial training |
| P03 Adversarial Patch | 1712.09665v2 §2 Eq.1 | patch / universal_eot | Bounded local and shared replacement EOT mechanisms implemented; original distribution/model/physical reproduction pending |
| P04 Vision-attack survey | 2410.23687v2 §7 | taxonomy | Reading map, not an executable attack |
| P05 Qi visual adversarial examples | 2306.13213v2 §3 Eq.1 | universal | Corpus-NLL adaptation implemented; original model/corpus/5000-step reproduction pending |
| P06 Jailbreak in Pieces | 2307.14539v2 §3.1 Eq.4 | feature | Qwen visual-feature Adam adaptation; original CLIP joint-space triggers pending |
| P07 Image Hijacks | 2309.00236v4 §2 Eq.1 | prompt_match | Teacher-forced soft-target matching implemented; original context-set/SGD reproduction pending |
| P08 BAP | 2406.04031v2 §4 Eqs.3–4 | bimodal | Corpus image plus private candidate-text search implemented; original feedback/LLM generator pending |
| P09 MMJ-Bench | AAAI 2025 DOI 10.1609/aaai.v39i26.34983 | comparison protocol | Benchmark importer and exact published configurations pending |
| P10 Physical Prompt Injection | 2601.17383v1 §§III–VI | injection | Real prebuilt image-artifact evaluation; physical generation/placement/camera tests pending |
| P11 AdaShield | 2403.09513v1 §3 | private defense_prefix | Static prompt comparison implemented; adaptive prompt-pool retrieval pending |
| P12 JailGuard | 2312.10766v4 §4 Eqs.4–7 | not implemented | Mathematics explained; original similarity encoder/mutators/detector not reproduced |

P01–P12 refer to the source-linked dossier embedded below. “Implemented” means code exists; it does not mean GPU validated or faithful reproduction. Surveys and benchmarks are not individual attacks. No full AdaShield-A or JailGuard reproduction is claimed.


## 2. Mathematical specification
    Let $f_\theta(x,t)$ be the fixed VLM, $x\in[0,1]^{3\times224\times224}$ the delivered attack canvas,
    $t$ the fixed user text, and $y^*$ a private target continuation.
    $L_{NLL}(x,t,y^*)=-\frac1{|y^*|}\sum_j\log p_\theta(y_j^*\mid x,t,y^*_{<j})$.
    Image, prompt and padding tokens do not contribute; causal prediction is shifted exactly once.

    **FGSM:** $x'=\Pi_{\mathcal C}(x-\epsilon\,sign(\nabla_x L_{NLL}))$ for targeted minimization.
    **PGD:** repeat $x_{k+1}=\Pi_{\mathcal C}(x_k-\alpha\,sign(\nabla L))$.
    Here $\mathcal C=[0,1]^d\cap B_\infty(x,\epsilon)$. Untargeted true-label maximization
    has the opposite sign; this real runner uses targeted generation loss, not classifier accuracy.

    **Localized patch:** $x'=(1-A)x+A p$, with a fixed binary top-left mask $A$;
    constrain $p$ to the original patch's epsilon interval. Record measured area.
    **Corpus image:** minimize the mean NLL over separate optimization contexts, one image shared across contexts.
    **Universal patch adaptation:** same shared context objective with a patch mask. It does not yet
    sample new background images, placements and rotations as Brown Eq.1 does.
    **Replacement EOT patch (`universal_eot`):** optimize one shared $p\in[0,1]$ across separate
    optimization backgrounds, seeded translations and right-angle rotations; deploy on a held-out background.
    This uses a box-only replacement threat model (`epsilon=1` inside the patch), not an imperceptible
    full-image budget. The small fixed transformation sample is recorded, not a physical camera simulation.

    **Feature matching:** $\min_{x'\in\mathcal C}\|V_\theta(x')-V_\theta(x_{target})\|_2^2$ via Adam.
    Current $V$ is Qwen's merged visual features, not original CLIP's joint embedding space.
    **Behavior matching:** minimize $-\sum_j q_j^T\log p_j(x',t)$ under the same teacher-forced
    continuation, with detached teacher distributions produced using a private teacher prompt.
    Full prompt-matching context generation in Image Hijacks remains to reproduce.

    **Bimodal adaptation:** first optimize the corpus image, then choose among 1–8 private text candidates
    by target NLL on development only. Save image-only, text-only and combined outputs. This does not
    reproduce BAP's feedback-driven ChatGPT candidate generation.
    **Injection:** evaluate a prebuilt instruction-bearing image against its paired clean image and fixed task.
    No gradient is required. Physical realism and placement optimization are not established.

    **Defenses:** compare private fixed-prefix generations on clean and attacked inputs. Optional flip-pair
    loss averaging implements a small EOT example, not a universal defense. JailGuard's mathematical
    pipeline is $S_{ij}=cos(v_i,v_j)$, $Q_i=S_i/\|S_i\|_1$, $D_{ij}=KL(Q_i\|Q_j)$;
    its published encoder, policy, thresholds and refusal handling still need implementation.


## 3. Embedded implementation — review before enabling GPU

All source hashes participate in the exact run identity. Raw media/text remain in private files. Image loading defines a 224-pixel square canvas before optimization; conclusions do not extend to perturbation budgets at the original source resolution.


In [2]:
SOURCE_HASHES = {'visual_core': 'fa67c3f928deb2c5f71fadec2f298e0594c6e5a627c37cb77aacd2a645999c36', 'visual_adapter': '274c8e8bbeb9557342f3199e0fc1dc8fb53b0db2aed4d32fbfabc06cd0f97455', 'visual_tests': '2ef71288b79d8da8d4352fdf7da1ef62690e7b6c42cdf724623598697cf9488d', 'analysis_plan': 'b7edf06001e782e4eac7659a698f7228d91759db5cf8cf0f274845bc7246bf42'}
import sys, types


In [3]:
visual_core_SOURCE = r'''
"""Real input optimizers and private runner. No model loading on import.

All methods minimize a supplied differentiable scalar objective. Original paper
settings and model-specific reproductions are separate from these adaptations.
"""
from dataclasses import asdict, dataclass
import hashlib
import json
import math
import os
from pathlib import Path
import time

import torch


def digest(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, allow_nan=False).encode()).hexdigest()


def atomic_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + '.tmp')
    tmp.write_text(json.dumps(value, sort_keys=True, allow_nan=False))
    os.replace(tmp, path)


@dataclass(frozen=True)
class AttackConfig:
    method: str = 'pgd'
    epsilon: float = 8/255
    steps: int = 20
    alpha: float = 2/255
    restarts: int = 1
    seed: int = 20261025
    patch_fraction: float = .04
    transform: str = 'identity'
    defense_aware: bool = False

    def validate(self):
        if self.method not in {'fgsm', 'pgd', 'patch', 'universal', 'universal_patch', 'universal_eot', 'feature', 'prompt_match', 'injection', 'bimodal'}:
            raise ValueError('Unsupported attack method')
        for v in (self.epsilon, self.alpha, self.patch_fraction):
            if not math.isfinite(v):
                raise ValueError('Nonfinite attack setting')
        if not 0 <= self.epsilon <= 1 or not 0 < self.alpha <= 1 or not 0 < self.patch_fraction <= 1:
            raise ValueError('Invalid attack bounds')
        if any(type(v) is not int or v < 1 for v in (self.steps, self.restarts)):
            raise ValueError('Positive integer steps/restarts required')
        if self.method == 'fgsm' and (self.steps != 1 or self.restarts != 1):
            raise ValueError('FGSM must have exactly one step and restart')
        if self.transform not in {'identity', 'flip_pair'}:
            raise ValueError('Unknown transform')
        if type(self.defense_aware) is not bool:
            raise ValueError('defense_aware must be boolean')
        if self.defense_aware and self.method in {'feature', 'injection'}:
            raise ValueError('This objective cannot optimize against a text defense')
        if self.method == 'universal_eot' and self.epsilon != 1:
            raise ValueError('Replacement EOT patch uses box-only pixels, epsilon=1; report its area')
        if self.method == 'universal_eot' and self.restarts != 1:
            raise ValueError('EOT patch adaptation supports one start; do not silently ignore restarts')


def pixels_valid(x):
    if x.ndim != 4 or x.shape[0] != 1 or x.shape[1] != 3 or not x.is_floating_point():
        raise ValueError('Expected one floating RGB NCHW image')
    if not torch.isfinite(x).all() or x.min() < 0 or x.max() > 1:
        raise ValueError('Pixels outside finite [0,1] range')


def patch_mask(x, fraction):
    h, w = x.shape[-2:]
    side = max(1, int(math.sqrt(fraction * h * w)))
    ph, pw = min(side, h), min(side, w)
    mask = torch.zeros_like(x)
    mask[..., :ph, :pw] = 1  # Fixed top-left location, recorded, never selected on test.
    return mask


def project(x, clean, epsilon, mask):
    low, high = (clean - epsilon).clamp(0, 1), (clean + epsilon).clamp(0, 1)
    bounded = torch.maximum(low, torch.minimum(high, x))
    return torch.where(mask.bool(), bounded, clean).detach()


def optimize(clean, objectives, config, deadline=None):
    """Minimize mean loss; universal modes share one image across private contexts.

    'feature' uses Adam; PGD variants use signed descent. Best iterate/restart is
    selected on optimization loss, never on held-out generations or guard labels.
    """
    config.validate()
    if config.method == 'injection':
        raise ValueError('Injection is an artifact evaluation, not gradient optimization')
    pixels_valid(clean)
    if not objectives:
        raise ValueError('At least one objective required')
    clean = clean.detach().clone()
    local = config.method in {'patch', 'universal_patch'}
    mask = patch_mask(clean, config.patch_fraction) if local else torch.ones_like(clean)
    best, best_loss, calls = clean.clone(), math.inf, 0
    trace = []

    def check_time():
        if deadline is not None and time.time() >= deadline:
            raise TimeoutError('Persistent run deadline reached')

    def score(x):
        nonlocal calls
        total = 0.
        with torch.no_grad():
            for objective in objectives:
                for view in ([x] if config.transform == 'identity' else [x, x.flip(-1)]):
                    check_time()
                    loss = objective(view)
                    calls += 1
                    if loss.ndim or not torch.isfinite(loss):
                        raise FloatingPointError('Invalid scalar objective')
                    total += float(loss)
        return total / (len(objectives) * (1 if config.transform == 'identity' else 2))

    baseline = score(clean)
    best_loss = baseline
    for restart in range(config.restarts):
        rng = torch.Generator(device=clean.device).manual_seed(config.seed + restart)
        if config.method == 'fgsm':
            x = clean.clone()
        else:
            noise = 2 * torch.rand(clean.shape, device=clean.device, dtype=clean.dtype, generator=rng) - 1
            x = project(clean + config.epsilon * noise, clean, config.epsilon, mask)
        adam = torch.optim.Adam([x.requires_grad_(True)], lr=config.alpha) if config.method == 'feature' else None
        for step in range(config.steps):
            check_time()
            x.requires_grad_(True)
            gradient = torch.zeros_like(x)
            count = len(objectives) * (1 if config.transform == 'identity' else 2)
            for objective in objectives:
                for flipped in ([False] if config.transform == 'identity' else [False, True]):
                    check_time()
                    loss = objective(x.flip(-1) if flipped else x)
                    calls += 1
                    if loss.ndim or not torch.isfinite(loss):
                        raise FloatingPointError('Invalid scalar objective')
                    grad, = torch.autograd.grad(loss, x)
                    if not torch.isfinite(grad).all():
                        raise FloatingPointError('Nonfinite image gradient')
                    gradient.add_(grad / count)
                    del loss, grad
            if gradient.abs().max() == 0 and config.epsilon > 0:
                raise FloatingPointError('Zero image gradient; attack not validated')
            if adam is not None:
                adam.zero_grad(set_to_none=True)
                x.grad = gradient
                adam.step()
                with torch.no_grad():
                    x.copy_(project(x, clean, config.epsilon, mask))
            else:
                step_size = config.epsilon if config.method == 'fgsm' else config.alpha
                x = project(x - step_size * gradient.sign(), clean, config.epsilon, mask)
            value = score(x)
            trace.append(dict(restart=restart, step=step+1, objective=value))
            if value < best_loss:
                best, best_loss = x.detach().clone(), value
    return best, dict(baseline_objective=baseline, best_objective=best_loss, forward_calls=calls,
                      trace=trace, effective_patch_fraction=float(mask.mean()) if local else None)


def quantize_bounded(candidate, clean, epsilon, mask=None):
    """Deliver valid uint8 pixels; reject budgets with no representable interval."""
    low = torch.ceil((clean - epsilon).clamp(0, 1) * 255)
    high = torch.floor((clean + epsilon).clamp(0, 1) * 255)
    if (low > high).any():
        raise ValueError('No representable pixel in declared budget')
    value = torch.maximum(low, torch.minimum(high, torch.round(candidate * 255))) / 255
    if mask is not None:
        value = torch.where(mask.bool(), value, clean)
    return value.detach()


def optimize_shared_patch(backgrounds, objectives, clean, config, deadline=None):
    """Shared replacement patch over backgrounds, translations and right-angle rotations.

    This is an EOT mechanism adaptation; it does not simulate a physical camera.
    Evaluation uses a fixed top-left placement and a background outside optimization.
    """
    config.validate()
    if config.method != 'universal_eot' or len(backgrounds) != len(objectives) or not backgrounds:
        raise ValueError('EOT requires paired optimization backgrounds/objectives')
    for image in backgrounds + [clean]:
        pixels_valid(image)
        if image.shape != clean.shape:
            raise ValueError('Shared patch needs equal canonical canvas shapes')
    mask = patch_mask(clean, config.patch_fraction)
    side = int(mask[0, 0].sum(1).max())
    h, w = clean.shape[-2:]
    rng = torch.Generator(device='cpu').manual_seed(config.seed)
    placements = [(int(torch.randint(h-side+1, (), generator=rng)),
                   int(torch.randint(w-side+1, (), generator=rng)),
                   int(torch.randint(4, (), generator=rng))) for _ in backgrounds]
    def compose(image, patch, placement):
        top, left, rotation = placement
        pad = (left, w-side-left, top, h-side-top)
        support = torch.nn.functional.pad(torch.ones_like(patch), pad)
        value = torch.nn.functional.pad(patch.rot90(rotation, (-2,-1)), pad)
        return image*(1-support)+value
    def check():
        if deadline is not None and time.time() >= deadline:
            raise TimeoutError('Deadline in shared-patch optimization')
    def score(patch):
        total = 0.
        with torch.no_grad():
            for image, objective, placement in zip(backgrounds, objectives, placements):
                check()
                loss = objective(compose(image, patch, placement))
                if loss.ndim or not torch.isfinite(loss):
                    raise FloatingPointError('Invalid EOT objective')
                total += float(loss)/len(backgrounds)
        return total
    patch = torch.full_like(clean[..., :side, :side], .5)  # No evaluation-background initialization.
    baseline = best_loss = score(patch)
    best, trace = patch.clone(), []
    for step in range(config.steps):
        patch = patch.detach().requires_grad_(True)
        gradient = torch.zeros_like(patch)
        for image, objective, placement in zip(backgrounds, objectives, placements):
            check()
            loss = objective(compose(image, patch, placement))
            if loss.ndim or not torch.isfinite(loss):
                raise FloatingPointError('Invalid EOT objective')
            grad, = torch.autograd.grad(loss, patch)
            gradient.add_(grad/len(backgrounds))
        if not torch.isfinite(gradient).all() or gradient.abs().max()==0:
            raise FloatingPointError('Invalid EOT gradient')
        patch = (patch-config.alpha*gradient.sign()).clamp(0,1).detach()
        value = score(patch)
        trace.append(dict(step=step+1, objective=value))
        if value < best_loss:
            best, best_loss = patch.clone(), value
    delivered = compose(clean, best, (0,0,0)).detach()
    return delivered, dict(baseline_objective=baseline, best_objective=best_loss, trace=trace,
            optimization_backgrounds=len(backgrounds), transformation_draws=len(placements),
            effective_patch_fraction=float(mask.mean()), evaluation_placement='top_left')


def validate_manifest(rows):
    if not isinstance(rows, list) or not rows:
        raise ValueError('Private manifest must be a nonempty list')
    ids, groups = set(), {}
    for row in rows:
        for key in ('id', 'image', 'image_sha256', 'prompt', 'target', 'split', 'intent_group', 'image_group', 'track'):
            if not isinstance(row.get(key), str) or not row[key]:
                raise ValueError('Missing or invalid manifest field: ' + key)
        if row['id'] in ids or row['split'] not in {'optimization', 'development', 'evaluation'}:
            raise ValueError('Duplicate ID or unknown split')
        if row['track'] not in {'perception', 'integrity', 'safety'}:
            raise ValueError('Unknown track')
        ids.add(row['id'])
        for key in ('intent_group', 'image_group'):
            group = (key, row[key])
            if group in groups and groups[group] != row['split']:
                raise ValueError('Group leakage across splits')
            groups[group] = row['split']
        exact_image = ('image_sha256', row['image_sha256'])
        if exact_image in groups and groups[exact_image] != row['split']:
            raise ValueError('Identical image content crosses splits')
        groups[exact_image] = row['split']
        raw = Path(row['image']).read_bytes()
        if hashlib.sha256(raw).hexdigest() != row['image_sha256']:
            raise ValueError('Image identity mismatch')
        if row['track'] == 'integrity' and 'injection_image' in row:
            expected = row.get('injection_sha256')
            if hashlib.sha256(Path(row['injection_image']).read_bytes()).hexdigest() != expected:
                raise ValueError('Injection image identity mismatch')
    return rows


def run_private(adapter, rows, config, out_dir, contract, deadline):
    """Paired generations, private artifacts, exact-resume keys; no automatic ASR."""
    from PIL import Image
    import numpy as np
    validate_manifest(rows)
    out = Path(out_dir)
    out.mkdir(parents=True, exist_ok=True)
    if not any(part == 'audit_work' or 'PRIVATE' in part for part in out.parts):
        raise ValueError('Outputs must be in an explicitly private directory')
    config.validate()
    clock = out / 'clock_PRIVATE.json'
    if clock.exists():
        saved = json.loads(clock.read_text())
        if saved['contract'] != digest(contract):
            raise ValueError('Run directory belongs to a different contract')
        deadline = min(deadline, saved['deadline'])
    else:
        atomic_json(clock, dict(contract=digest(contract), deadline=deadline))
    counts = dict(completed=0, cached=0, cached_failed=0, failed=0, stopped=0)
    def generate_checked(image, text):
        if time.time() >= deadline:
            raise TimeoutError('Deadline before generation')
        return adapter.generate(image, text)

    for row in rows:
        if row['split'] != 'development':
            continue  # Reserved evaluation cannot be opened by this development runner.
        if time.time() >= deadline:
            counts['stopped'] += 1
            break
        identity = dict(contract=contract, row=row, attack=asdict(config), adapter=adapter.identity)
        key = digest(identity)
        dest = out / (key + '_PRIVATE.json')
        if dest.exists():
            cached = json.loads(dest.read_text())
            if cached.get('key') != key or cached.get('status') not in {'complete', 'failed'}:
                raise ValueError('Corrupt cache identity')
            if cached['status'] == 'complete':
                artifact = out / (key + '_PRIVATE.png')
                if cached.get('identity') != identity or not artifact.exists() or hashlib.sha256(artifact.read_bytes()).hexdigest() != cached['artifact_sha256']:
                    raise ValueError('Completed cache or artifact identity changed')
            counts['cached' if cached['status'] == 'complete' else 'cached_failed'] += 1
            continue
        start = time.monotonic()
        responses = {}
        try:
            clean = adapter.read_image(row['image'])
            parity = adapter.parity_check(clean, row['prompt'], row['target'])
            if time.time() >= deadline:
                raise TimeoutError('Deadline after parity gate')
            weights_before = adapter.frozen_fingerprint()
            defense = row.get('defense_prefix', '')
            if config.defense_aware and not defense:
                raise ValueError('Defense-aware optimization requires a fixed defense prefix')
            optimization_prompt = (defense if config.defense_aware else '') + row['prompt']
            target_loss = adapter.objective(clean, optimization_prompt, row['target'])
            if config.method != 'injection':
                initial = clean.clone().requires_grad_(True)
                grad, = torch.autograd.grad(target_loss(initial), initial)
                if not torch.isfinite(grad).all() or grad.abs().max() == 0:
                    raise FloatingPointError('Gradient gate failed')
                del initial, grad
            if config.method == 'injection':
                if not row.get('injection_image'):
                    raise ValueError('Injection artifact required')
                objectives = []
            elif config.method in {'universal', 'universal_patch', 'universal_eot', 'bimodal'}:
                corpus = [r for r in rows if r['split'] == 'optimization']
                if not corpus:
                    raise ValueError('Universal method requires separate optimization contexts')
                objectives = [adapter.objective(clean, (defense if config.defense_aware else '')+r['prompt'], r['target']) for r in corpus]
            elif config.method == 'feature':
                if 'feature_image' not in row or 'feature_sha256' not in row:
                    raise ValueError('Feature target artifact is required')
                if hashlib.sha256(Path(row['feature_image']).read_bytes()).hexdigest() != row['feature_sha256']:
                    raise ValueError('Feature target identity mismatch')
                objectives = [adapter.feature_objective(adapter.read_image(row['feature_image']))]
            elif config.method == 'prompt_match':
                if not row.get('teacher_prompt'):
                    raise ValueError('Private teacher_prompt required')
                objectives = [adapter.match_objective(clean, optimization_prompt, row['target'], row['teacher_prompt'])]
            else:
                objectives = [target_loss]
            if config.method == 'injection':
                attacked, metrics = clean.clone(), dict(gradient_optimization=False)
            elif config.method == 'universal_eot':
                backgrounds = [adapter.read_image(r['image']) for r in corpus]
                attacked, metrics = optimize_shared_patch(backgrounds, objectives, clean, config, deadline)
            else:
                attacked, metrics = optimize(clean, objectives, config, deadline)
            mask = patch_mask(clean, config.patch_fraction) if config.method in {'patch', 'universal_patch', 'universal_eot'} else None
            attacked = quantize_bounded(attacked, clean, config.epsilon, mask)
            artifact = out / (key + '_PRIVATE.png')
            arr = (attacked[0].permute(1, 2, 0).cpu().numpy() * 255).round().astype(np.uint8)
            tmp = artifact.with_suffix('.tmp')
            Image.fromarray(arr).save(tmp, format='PNG')
            os.replace(tmp, artifact)
            delivered = adapter.read_image(artifact)
            if (delivered - clean).abs().max() > config.epsilon + 1e-6:
                raise ValueError('Delivered artifact exceeds raw-pixel budget')
            if mask is not None and not torch.equal(delivered[mask == 0], clean[mask == 0]):
                raise ValueError('Delivered patch changed pixels outside mask')
            active_prompt = row['prompt']
            candidate_evaluations = 0
            if config.method == 'bimodal':
                candidates = row.get('candidate_prompts', [])
                if not candidates or len(candidates) > 8 or any(not isinstance(p, str) or not p for p in candidates):
                    raise ValueError('Bimodal adaptation needs 1..8 private candidate prompts')
                values = []
                with torch.no_grad():
                    for prompt in [row['prompt']] + candidates:
                        if time.time() >= deadline:
                            raise TimeoutError('Deadline during text candidate search')
                        values.append(float(adapter.objective(clean, (defense if config.defense_aware else '')+prompt, row['target'])(delivered)))
                active_prompt = ([row['prompt']] + candidates)[min(range(len(values)), key=values.__getitem__)]
                candidate_evaluations = len(values)
            responses = {}
            for name, image in [('clean', clean), ('attack', delivered)]:
                if time.time() >= deadline:
                    raise TimeoutError('Deadline before paired generation')
                responses[name] = generate_checked(image, row['prompt'])
            responses['text_only'] = generate_checked(None, row['prompt'])
            rng = torch.Generator(device=clean.device).manual_seed(config.seed)
            noise = (torch.rand(clean.shape, device=clean.device, dtype=clean.dtype, generator=rng)*2-1)*config.epsilon
            random_image = quantize_bounded((clean+noise).clamp(0,1), clean, config.epsilon)
            if mask is not None:
                random_image = torch.where(mask.bool(), random_image, clean)
            responses['random_control'] = generate_checked(random_image, row['prompt'])
            if active_prompt != row['prompt']:
                if time.time() >= deadline:
                    raise TimeoutError('Deadline before combined generation')
                responses['combined'] = generate_checked(delivered, active_prompt)
                responses['text_candidate_only'] = generate_checked(clean, active_prompt)
            if row.get('injection_image'):
                if time.time() >= deadline:
                    raise TimeoutError('Deadline before injection generation')
                injection = adapter.read_image(row['injection_image'])
                responses['injection'] = generate_checked(injection, row['prompt'])
            if row.get('defense_prefix'):
                if time.time() >= deadline:
                    raise TimeoutError('Deadline before defense generation')
                defended_image = injection if config.method == 'injection' else delivered
                responses['defended_attack'] = generate_checked(defended_image, row['defense_prefix'] + active_prompt)
                if time.time() >= deadline:
                    raise TimeoutError('Deadline before clean defense generation')
                responses['defended_clean'] = generate_checked(clean, row['defense_prefix'] + row['prompt'])
            if adapter.frozen_fingerprint() != weights_before:
                raise ValueError('Model weights changed during input optimization')
            atomic_json(dest, dict(key=key, status='complete', identity=identity, responses=responses,
                                  metrics=metrics, parity=parity, selected_prompt=active_prompt,
                                  candidate_evaluations=candidate_evaluations, elapsed_seconds=time.monotonic()-start,
                                  artifact_sha256=hashlib.sha256(artifact.read_bytes()).hexdigest(),
                                  delivered_linf=float((delivered-clean).abs().max())))
            counts['completed'] += 1
        except Exception as error:
            # Error messages can contain prompts: store only type and never print replies.
            atomic_json(dest, dict(key=key, status='failed', error_type=type(error).__name__,
                                  partial_responses=responses, elapsed_seconds=time.monotonic()-start))
            counts['failed'] += 1
            if isinstance(error, (TimeoutError, torch.OutOfMemoryError)) or 'device-side' in str(error):
                counts['stopped'] += 1
                break
    atomic_json(out / 'aggregate_status.json', dict(**counts, contract_hash=digest(contract),
                safety_labels='not_collected', claim_status='development_only', deadline=deadline))
    return counts
'''
visual_core = types.ModuleType('visual_core')
sys.modules['visual_core'] = visual_core
exec(compile(visual_core_SOURCE, 'visual_core', "exec"), visual_core.__dict__)


/Users/aryamandev/.pyenv/versions/3.11.6/lib/python3.11/site-packages/torch/cuda/__init__.py:65: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


In [4]:
visual_adapter_SOURCE = r'''
"""Differentiable Qwen2.5-VL adapter, batch one, native Transformers, frozen weights.

The attack canvas is an explicit 224x224 RGB image. Source-image resizing occurs
BEFORE defining its perturbation budget; original-resolution robustness is not claimed.
Patch layout follows the Apache-2.0 Transformers Qwen2VLImageProcessor implementation:
https://github.com/huggingface/transformers/blob/v4.57.1/src/transformers/models/qwen2_vl/image_processing_qwen2_vl.py
No upstream source copied; processor parity is checked before optimization.
"""
import hashlib
import time

import torch
import torch.nn.functional as F


def patchify(raw, image_processor):
    """Single still image, normalized then temporal-repeat and spatial patch grouping."""
    if raw.ndim != 4 or raw.shape[:2] != (1, 3):
        raise ValueError('Single RGB still image required')
    p, m, t = image_processor.patch_size, image_processor.merge_size, image_processor.temporal_patch_size
    h, w = raw.shape[-2:]
    if h % (p*m) or w % (p*m):
        raise ValueError('Canvas dimensions must be multiples of patch*merge')
    mean = raw.new_tensor(image_processor.image_mean).view(1, 3, 1, 1)
    std = raw.new_tensor(image_processor.image_std).view(1, 3, 1, 1)
    z = (raw - mean) / std
    gh, gw = h//p, w//p
    z = z.repeat(t, 1, 1, 1).reshape(1, t, 3, gh//m, m, p, gw//m, m, p)
    return z.permute(0, 3, 6, 4, 7, 2, 1, 5, 8).reshape(gh*gw, 3*t*p*p)


def shifted_nll(logits, ids, target_start):
    if not 1 <= target_start < ids.shape[1]:
        raise ValueError('Empty/misaligned target span')
    selected = logits[:, target_start-1:-1].float()
    targets = ids[:, target_start:]
    return F.cross_entropy(selected.reshape(-1, selected.shape[-1]), targets.reshape(-1))


class QwenAttackAdapter:
    MODEL = 'Qwen/Qwen2.5-VL-3B-Instruct'
    REVISION = '66285546d2b821cf421d4f5eb2576359d3770cd3'

    def __init__(self, canvas=224, max_new=256):
        import transformers
        from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration
        if transformers.__version__ != '4.57.1':
            raise RuntimeError('Use reviewed transformers==4.57.1; restart after installing')
        if not torch.cuda.is_available():
            raise RuntimeError('Real VLM branch requires CUDA; CPU checks use injected tiny models')
        if canvas != 224 or max_new < 1 or max_new > 512:
            raise ValueError('Reviewed canvas=224; output cap must be 1..512')
        self.device, self.canvas, self.max_new = torch.device('cuda:0'), canvas, max_new
        self.processor = AutoProcessor.from_pretrained(self.MODEL, revision=self.REVISION,
                trust_remote_code=False, use_fast=False, min_pixels=canvas**2, max_pixels=canvas**2)
        # Explicit single device. No silent CPU offload or quantization substitutions.
        self.model = Qwen2_5_VLForConditionalGeneration.from_pretrained(self.MODEL,
                revision=self.REVISION, trust_remote_code=False, torch_dtype=torch.float16,
                attn_implementation='eager').to(self.device).eval()
        self.model.requires_grad_(False)
        self.model.config.use_cache = False
        self.identity = dict(model=self.MODEL, revision=self.REVISION, canvas=canvas,
                precision='fp16', attention='eager', batch=1, max_new=max_new,
                transformers=transformers.__version__, torch=torch.__version__,
                image_processor=self.processor.image_processor.to_dict(), generation='greedy')

    def read_image(self, path):
        from PIL import Image
        import numpy as np
        im = Image.open(path).convert('RGB')
        # Budget is on this canonical delivered canvas, never on original source pixels.
        if im.size != (self.canvas, self.canvas):
            im = im.resize((self.canvas, self.canvas), Image.Resampling.BICUBIC)
        return torch.from_numpy(np.array(im, copy=True)).permute(2, 0, 1).unsqueeze(0).to(self.device).float()/255

    def _inputs(self, image, prompt, target=None):
        from PIL import Image
        import numpy as np
        # Only template/IDs/grid go through non-differentiable processor. Replace
        # pixel_values with patchify(image) in _forward to keep raw input gradients.
        dummy = Image.fromarray(np.zeros((self.canvas, self.canvas, 3), dtype=np.uint8))
        content = ([dict(type='image')] if image is not None else []) + [dict(type='text', text=prompt)]
        messages = [dict(role='user', content=content)]
        text = self.processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        data = self.processor(text=[text], images=[dummy] if image is not None else None, return_tensors='pt')
        data = {k: v.to(self.device) for k, v in data.items()}
        data.pop('pixel_values', None)
        start = data['input_ids'].shape[1]
        if target is not None:
            target_ids = self.processor.tokenizer(target, add_special_tokens=False, return_tensors='pt')['input_ids'].to(self.device)
            if not 0 < target_ids.shape[1] <= 128:
                raise ValueError('Target must have 1..128 tokens')
            data['input_ids'] = torch.cat([data['input_ids'], target_ids], dim=1)
            data['attention_mask'] = torch.ones_like(data['input_ids'])
        if data['input_ids'].shape[1] > 1024:
            raise ValueError('Development input token cap exceeded')
        return data, start

    def _forward(self, image, inputs):
        return self.model(**inputs, pixel_values=patchify(image, self.processor.image_processor).half(),
                          use_cache=False).logits

    def objective(self, clean, prompt, target):
        inputs, start = self._inputs(clean, prompt, target)
        def loss(image):
            return shifted_nll(self._forward(image, inputs), inputs['input_ids'], start)
        return loss

    def feature_objective(self, target_image):
        # Qwen merged visual-token features, not CLIP joint text/image embeddings.
        grid = torch.tensor([[1, self.canvas//14, self.canvas//14]], device=self.device)
        def features(image):
            value = self.model.model.visual(patchify(image, self.processor.image_processor).half(), grid_thw=grid)
            if not isinstance(value, torch.Tensor):
                raise TypeError('Unexpected Qwen visual output; inspect new library before use')
            return value.float()
        with torch.no_grad():
            target = features(target_image).detach()
        return lambda image: F.mse_loss(features(image), target)

    def match_objective(self, clean, prompt, target, teacher_prompt):
        student, start = self._inputs(clean, prompt, target)
        teacher, teacher_start = self._inputs(clean, teacher_prompt, target)
        with torch.no_grad():
            distribution = self._forward(clean, teacher)[:, teacher_start-1:-1].float().softmax(-1).detach()
        def loss(image):
            logits = self._forward(image, student)[:, start-1:-1].float()
            # Soft cross entropy under SAME teacher-forced continuation; adaptation
            # of behavioral matching, not the original full context-set algorithm.
            return -(distribution * logits.log_softmax(-1)).sum(-1).mean()
        return loss

    def parity_check(self, image, prompt, target):
        import numpy as np
        from PIL import Image
        pil = Image.fromarray((image[0].detach().cpu().permute(1, 2, 0).numpy()*255).round().astype(np.uint8))
        ref = self.processor.image_processor(images=[pil], return_tensors='pt')
        ours = patchify(image, self.processor.image_processor)
        reference = ref['pixel_values'].to(ours.device)
        if ours.shape != reference.shape or not torch.allclose(ours, reference, atol=2e-6, rtol=2e-6):
            raise ValueError('Differentiable processor parity failed')
        inputs, start = self._inputs(image, prompt, target)
        if not torch.equal(inputs['image_grid_thw'], ref['image_grid_thw'].to(self.device)):
            raise ValueError('Image grid differs from reference')
        with torch.no_grad():
            actual = self._forward(image, inputs)[:, start-1].float()
            expected = self.model(**inputs, pixel_values=reference.half(), use_cache=False).logits[:, start-1].float()
        if not torch.isfinite(actual).all() or not torch.allclose(actual, expected, atol=.05, rtol=.01):
            raise ValueError('Clean logits differ from reference')
        return dict(processor_max_error=float((ours-reference).abs().max()), logits_max_error=float((actual-expected).abs().max()))

    def generate(self, image, prompt):
        inputs, start = self._inputs(image, prompt)
        torch.cuda.reset_peak_memory_stats()
        begin = time.monotonic()
        visual = {} if image is None else dict(pixel_values=patchify(image, self.processor.image_processor).half())
        with torch.no_grad():
            result = self.model.generate(**inputs, **visual,
                    max_new_tokens=self.max_new, do_sample=False, use_cache=True)
        torch.cuda.synchronize()
        ids = result[0, start:]
        eos = self.model.generation_config.eos_token_id
        eos = eos if isinstance(eos, list) else [eos]
        return dict(response=self.processor.tokenizer.decode(ids, skip_special_tokens=True),
                    generated_ids=ids.cpu().tolist(), cap_hit=len(ids) >= self.max_new and int(ids[-1]) not in eos,
                    seconds=time.monotonic()-begin, peak_allocated_gb=torch.cuda.max_memory_allocated()/1e9,
                    peak_reserved_gb=torch.cuda.max_memory_reserved()/1e9)

    def frozen_fingerprint(self):
        # Full weight fingerprint is expensive; check frozen flags and mutation version
        # counters for ALL tensors, plus representative values. Loading is revision pinned.
        states = []
        for name, p in self.model.named_parameters():
            if p.requires_grad:
                raise ValueError('Trainable model parameter detected')
            sample = p.detach().reshape(-1)[:4].float().cpu().numpy().tobytes()
            states.append((name, p._version, hashlib.sha256(sample).hexdigest()))
        return states

    def close(self):
        self.model = None
        import gc
        gc.collect()
        torch.cuda.empty_cache()
'''
visual_adapter = types.ModuleType('visual_adapter')
sys.modules['visual_adapter'] = visual_adapter
exec(compile(visual_adapter_SOURCE, 'visual_adapter', "exec"), visual_adapter.__dict__)


In [5]:
visual_tests_SOURCE = r'''
"""Offline acceptance checks; no pretrained weights, benchmark data or GPU needed."""
import hashlib
import json
from pathlib import Path
import tempfile
import time
from types import SimpleNamespace

import numpy as np
from PIL import Image
import torch
import torch.nn.functional as F


def run_cpu_checks(core, adapter_module):
    torch.set_num_threads(1)
    clean = torch.full((1, 3, 28, 28), 128/255, dtype=torch.float64)
    objective = lambda x: (x.mean() - .2).square()
    losses = {}
    for method in ('fgsm', 'pgd', 'patch', 'universal', 'universal_patch', 'feature', 'prompt_match', 'bimodal'):
        cfg = core.AttackConfig(method=method, epsilon=.1, alpha=.03,
                steps=1 if method == 'fgsm' else 3, patch_fraction=.04)
        objectives = [objective, lambda x: (x.mean()-.3).square()] if method.startswith('universal') else [objective]
        result, metrics = core.optimize(clean, objectives, cfg)
        assert (result-clean).abs().max() <= cfg.epsilon + 1e-12
        assert result.min() >= 0 and result.max() <= 1
        assert metrics['best_objective'] < metrics['baseline_objective']
        if 'patch' in method:
            mask = core.patch_mask(clean, cfg.patch_fraction)
            assert torch.equal(result[mask == 0], clean[mask == 0])
        quantized = core.quantize_bounded(result, clean, cfg.epsilon)
        assert (quantized-clean).abs().max() <= cfg.epsilon + 1e-12
        losses[method] = metrics['best_objective']
    cfg = core.AttackConfig(steps=2, restarts=2, transform='flip_pair')
    eot_cfg = core.AttackConfig(method='universal_eot',epsilon=1,steps=3,alpha=.1)
    eot, eot_metrics = core.optimize_shared_patch([clean,clean*.9],[objective,objective],clean,eot_cfg)
    assert eot_metrics['best_objective'] < eot_metrics['baseline_objective']
    eot_mask = core.patch_mask(clean,eot_cfg.patch_fraction)
    assert torch.equal(eot[eot_mask==0],clean[eot_mask==0])
    a, _ = core.optimize(clean, [objective], cfg)
    b, _ = core.optimize(clean, [objective], cfg)
    assert torch.equal(a, b)
    try:
        core.optimize(clean, [objective], cfg, deadline=time.time()-1)
    except TimeoutError:
        pass
    else:
        raise AssertionError('Expired deadline accepted')
    for bad in [dict(epsilon=-1), dict(steps=0), dict(alpha=float('nan')), dict(method='missing')]:
        try:
            core.AttackConfig(**bad).validate()
        except ValueError:
            pass
        else:
            raise AssertionError('Invalid configuration accepted')
    try:
        core.optimize(clean, [lambda x: x.mean()*float('nan')], cfg)
    except FloatingPointError:
        pass
    else:
        raise AssertionError('Nonfinite loss accepted')

    proc = SimpleNamespace(patch_size=14, merge_size=2, temporal_patch_size=2,
            image_mean=[.48145466, .4578275, .40821073], image_std=[.26862954, .26130258, .27577711])
    raw = torch.linspace(0, 1, 3*56*28, dtype=torch.float32).reshape(1, 3, 56, 28).requires_grad_(True)
    actual = adapter_module.patchify(raw, proc)
    z = (raw.detach().numpy()-np.array(proc.image_mean, dtype=np.float32).reshape(1,3,1,1))/np.array(proc.image_std,dtype=np.float32).reshape(1,3,1,1)
    reference = np.repeat(z, 2, axis=0).reshape(1,2,3,2,2,14,1,2,14).transpose(0,3,6,4,7,2,1,5,8).reshape(8,1176)
    assert np.allclose(actual.detach().numpy(), reference, atol=2e-6)
    gradient, = torch.autograd.grad(actual.square().mean(), raw)
    assert torch.isfinite(gradient).all() and gradient.abs().max() > 0
    logits = torch.randn(1, 6, 7, generator=torch.Generator().manual_seed(1), requires_grad=True)
    ids = torch.tensor([[1,2,3,4,5,6]])
    value = adapter_module.shifted_nll(logits, ids, 4)
    expected = F.cross_entropy(logits[:,3:5].reshape(-1,7), ids[:,4:].reshape(-1))
    assert torch.allclose(value, expected)
    value.backward()
    assert logits.grad[:,:3].abs().sum() == 0 and logits.grad[:,3:5].abs().sum() > 0
    assert logits.grad[:,5:].abs().sum() == 0

    class FakeAdapter:
        identity = dict(model='CPU_STUB', revision='offline')
        def read_image(self, path):
            im = np.asarray(Image.open(path).convert('RGB'), dtype=np.float64)/255
            return torch.tensor(im.transpose(2,0,1)[None])
        def parity_check(self, image, prompt, target):
            return dict(processor_max_error=0.)
        def frozen_fingerprint(self):
            return [('frozen', 0)]
        def objective(self, clean, prompt, target):
            if target == 'FAIL_TEST':
                return lambda x: x.mean()*float('nan')
            return objective
        def feature_objective(self, target):
            return objective
        def match_objective(self, clean, prompt, target, teacher_prompt):
            return objective
        def generate(self, image, prompt):
            return dict(response='PRIVATE_TEST_SENTINEL', cap_hit=False)

    with tempfile.TemporaryDirectory() as temp:
        root = Path(temp)
        image = root/'input.png'
        Image.fromarray(np.full((28,28,3),128,dtype=np.uint8)).save(image)
        row = dict(id='dev', image=str(image), image_sha256=hashlib.sha256(image.read_bytes()).hexdigest(),
                prompt='PRIVATE_TEST_SENTINEL', target='TEST_TARGET', intent_group='d', image_group='d',
                track='integrity', split='development', injection_image=str(image),
                injection_sha256=hashlib.sha256(image.read_bytes()).hexdigest(),
                feature_image=str(image), feature_sha256=hashlib.sha256(image.read_bytes()).hexdigest(),
                teacher_prompt='TEST_TEACHER', candidate_prompts=['TEST_CANDIDATE'], defense_prefix='TEST_DEFENSE')
        opt_image = root/'optimization.png'
        Image.fromarray(np.full((28,28,3),127,dtype=np.uint8)).save(opt_image)
        opt = dict(row, id='opt', split='optimization', intent_group='o', image_group='o',
                image=str(opt_image),image_sha256=hashlib.sha256(opt_image.read_bytes()).hexdigest())
        for method in ('fgsm','pgd','patch','universal','universal_patch','universal_eot','feature','prompt_match','injection','bimodal'):
            out = root/(method+'_PRIVATE')
            cfg = core.AttackConfig(method=method, steps=1 if method=='fgsm' else 2,
                    epsilon=1 if method=='universal_eot' else 8/255)
            contract = dict(test=method)
            counts = core.run_private(FakeAdapter(), [row,opt], cfg, out, contract, time.time()+60)
            assert counts['completed']==1 and counts['failed']==0
            counts = core.run_private(FakeAdapter(), [row,opt], cfg, out, contract, time.time()+60)
            assert counts['cached']==1 and counts['completed']==0
            export = json.loads((out/'aggregate_status.json').read_text())
            assert 'PRIVATE_TEST_SENTINEL' not in json.dumps(export)
            try:
                core.run_private(FakeAdapter(),[row,opt],cfg,out,dict(test='changed'),time.time()+60)
            except ValueError:
                pass
            else:
                raise AssertionError('Changed contract reused run directory')
        reserved = dict(row,id='eval',split='evaluation',intent_group='e',image_group='e')
        out = root/'reserved_PRIVATE'
        assert core.run_private(FakeAdapter(),[reserved],cfg,out,{},time.time()+60)['completed']==0
        try:
            core.validate_manifest([row,dict(row,id='leak',split='evaluation')])
        except ValueError:
            pass
        else:
            raise AssertionError('Group leakage accepted')
        bad = dict(row,target='FAIL_TEST')
        out = root/'failure_PRIVATE'
        counts = core.run_private(FakeAdapter(),[bad],core.AttackConfig(),out,{},time.time()+60)
        assert counts['failed']==1
        assert core.run_private(FakeAdapter(),[bad],core.AttackConfig(),out,{},time.time()+60)['cached_failed']==1
        changed = dict(row,image_sha256='invalid')
        try:
            core.validate_manifest([changed])
        except ValueError:
            pass
        else:
            raise AssertionError('Changed image accepted')
    return dict(optimizer_methods=len(losses)+1, private_runner_methods=10, processor_layout='passed',
                causal_target_mask='passed', resume_failure_privacy='passed', gpu='not_tested')


def run_architecture_check(adapter_module):
    """Exact reviewed native API, randomly initialized tiny weights; no downloads."""
    import transformers
    from transformers import Qwen2_5_VLConfig, Qwen2_5_VLForConditionalGeneration
    from transformers.models.qwen2_vl.image_processing_qwen2_vl import Qwen2VLImageProcessor
    if transformers.__version__ != '4.57.1':
        raise RuntimeError('Architecture check requires transformers==4.57.1')
    torch.set_num_threads(1)
    raw = (torch.arange(3*56*56).reshape(1,3,56,56)%256).float()/255
    proc = Qwen2VLImageProcessor(min_pixels=56*56,max_pixels=56*56)
    image = Image.fromarray((raw[0].permute(1,2,0).numpy()*255).round().astype(np.uint8))
    reference = proc(images=[image],return_tensors='pt')['pixel_values']
    raw.requires_grad_(True)
    ours = adapter_module.patchify(raw,proc)
    assert torch.allclose(ours,reference,atol=2e-6,rtol=2e-6)
    config = Qwen2_5_VLConfig(text_config=dict(vocab_size=32,hidden_size=32,intermediate_size=64,
            num_hidden_layers=1,num_attention_heads=4,num_key_value_heads=2,
            rope_scaling={'type':'mrope','mrope_section':[2,1,1]}),
            vision_config=dict(depth=1,hidden_size=32,intermediate_size=64,num_heads=4,
            out_hidden_size=32,patch_size=14,spatial_merge_size=2,temporal_patch_size=2,
            window_size=28,fullatt_block_indexes=[0]),image_token_id=5,video_token_id=4,
            vision_start_token_id=6,vision_end_token_id=7)
    config._attn_implementation = 'eager'
    with torch.random.fork_rng():
        torch.manual_seed(1)
        model = Qwen2_5_VLForConditionalGeneration(config).eval().requires_grad_(False)
    ids = torch.tensor([[1,6,5,5,5,5,7,8,9]])
    grid = torch.tensor([[1,4,4]])
    out = model(input_ids=ids,attention_mask=torch.ones_like(ids),pixel_values=ours,
                image_grid_thw=grid,use_cache=False)
    gradient, = torch.autograd.grad(adapter_module.shifted_nll(out.logits,ids,8),raw)
    assert torch.isfinite(gradient).all() and gradient.abs().max()>0
    assert isinstance(model.model.visual(ours,grid_thw=grid),torch.Tensor)
    return dict(native_api=transformers.__version__,preprocessor='passed',
                native_image_gradient='passed',pretrained_weights=False,gpu=False)


if __name__ == '__main__':
    import sys
    sys.path.insert(0,str(Path(__file__).resolve().parents[2]))
    import vlm_visual_attacks as core_module
    import vlm_visual_adapter as adapter_module
    print(run_cpu_checks(core_module,adapter_module))
    if '--architecture' in sys.argv:
        print(run_architecture_check(adapter_module))
'''
visual_tests = types.ModuleType('visual_tests')
sys.modules['visual_tests'] = visual_tests
exec(compile(visual_tests_SOURCE, 'visual_tests', "exec"), visual_tests.__dict__)


In [6]:
CPU_CHECKS = visual_tests.run_cpu_checks(visual_core, visual_adapter)
print(CPU_CHECKS)


{'optimizer_methods': 9, 'private_runner_methods': 10, 'processor_layout': 'passed', 'causal_target_mask': 'passed', 'resume_failure_privacy': 'passed', 'gpu': 'not_tested'}


### Readout — editable

These acceptance tests use analytic objectives and a fake adapter, never a pretrained VLM. They cover all runner paths, saved bounds, target masking, group leakage, identity, deadlines, failure caches and numeric-only summaries. T4 memory and real model behavior remain untested.


## 4. Private data contract

MANIFEST_PATH points to a private JSON list. Required row fields: `id,image,image_sha256,prompt,target,split,intent_group,image_group,track`. Splits: `optimization/development/evaluation`; tracks: `perception/integrity/safety`. No template embeds actual prompts or replies. For injection add `injection_image,injection_sha256`; feature matching uses `feature_image,feature_sha256`; behavior matching uses `teacher_prompt`; bimodal search uses `candidate_prompts`; static defense uses `defense_prefix`. Paths are absolute private runtime paths. Group by underlying image and intent across all variants. The first runner processes at most four development rows and never opens reserved evaluation outcomes. Universal optimization requires separate optimization rows. Artifact and group checks run before loading weights.


In [7]:
import time
from pathlib import Path
from dataclasses import asdict
import hashlib
import json

config = visual_core.AttackConfig(method=METHOD, steps=1 if METHOD=='fgsm' else 10,
                                 epsilon=1 if METHOD=='universal_eot' else 8/255,
                                 defense_aware=DEFENSE_AWARE)
config.validate()
run_result = None
if RUN_GPU:
    if not COMPUTE_APPROVED or not SCIENTIFIC_AMENDMENT_ACCEPTED:
        raise RuntimeError('Review and accept the dated amendment and this compute scope first')
    if not MANIFEST_PATH or not OUT_DIR or not 1 <= MAX_WALL_MINUTES <= 30:
        raise ValueError('Private manifest/output and a 1..30 minute development budget required')
    manifest_bytes = Path(MANIFEST_PATH).read_bytes()
    rows = visual_core.validate_manifest(json.loads(manifest_bytes))
    scope = dict(run_id=RUN_ID, source_hashes=SOURCE_HASHES, method=asdict(config),
            manifest_sha256=hashlib.sha256(manifest_bytes).hexdigest(), max_minutes=MAX_WALL_MINUTES,
            model=visual_adapter.QwenAttackAdapter.MODEL,
            revision=visual_adapter.QwenAttackAdapter.REVISION, canvas=224, max_new=MAX_NEW_TOKENS)
    scope_hash = visual_core.digest(scope)
    if APPROVED_SCOPE_SHA256 != scope_hash:
        raise RuntimeError('Record this exact approved scope hash: ' + scope_hash)
    if not any(r['split']=='development' for r in rows):
        raise ValueError('No development rows; reserved evaluation is locked')
    if len([r for r in rows if r['split']=='development']) > 4:
        raise ValueError('First real stage capped at four development items')
    if len([r for r in rows if r['split']=='optimization']) > 8:
        raise ValueError('First real stage capped at eight optimization contexts')
    if not Path('/content/drive/MyDrive').is_dir():
        if not MOUNT_DRIVE:
            raise RuntimeError('Mount college Drive explicitly before starting the bounded run')
        from google.colab import drive
        drive.mount('/content/drive')  # Owner handles Google permission interaction.
    root = Path(OUT_DIR).resolve()
    base = Path('/content/drive/MyDrive').resolve()
    if base not in root.parents or 'PRIVATE' not in root.name:
        raise ValueError('Use a distinct PRIVATE output folder inside mounted college Drive')
    root.mkdir(parents=True, exist_ok=True)
    clock = root/'clock_PRIVATE.json'
    deadline = time.time() + MAX_WALL_MINUTES*60
    if clock.exists():
        state = json.loads(clock.read_text())
        if state['contract'] != scope_hash:
            raise ValueError('Output directory has a different experiment identity')
        deadline = min(deadline, state['deadline'])
    else:
        visual_core.atomic_json(clock, dict(contract=scope_hash, deadline=deadline))
    if time.time() >= deadline:
        raise TimeoutError('Original time budget expired; preserve partial outputs')
else:
    print('GPU OFF: no installations, Drive mounts, model downloads or inference.')


GPU OFF: no installations, Drive mounts, model downloads or inference.


### Readout — editable

The source/plan/manifest/config hash is the exact scope to review. Set it only after scope approval. The persisted deadline includes installations and downloads; a resumed run cannot reset it. Do not delete a previous clock or repurpose its folder to extend a stage.


In [8]:
if RUN_GPU:
    import importlib.metadata
    import subprocess
    import sys
    if INSTALL_REVIEWED_DEPENDENCIES:
        if time.time() >= deadline:
            raise TimeoutError('Deadline before installation')
        subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet',
                        'transformers==4.57.1', 'Pillow==11.3.0'],
                        check=True, timeout=max(1, int(deadline-time.time())))
        if importlib.metadata.version('transformers') != '4.57.1':
            raise RuntimeError('Dependency installation did not select reviewed version')
    print('Dependency stage complete. If transformers was already imported, restart and resume; clock persists.')


### Readout — editable

No installation occurs with GPU off. The real adapter refuses unreviewed Transformers versions. Restart if an old version was already imported; outputs and the deadline stay on Drive.


In [9]:
if RUN_GPU:
    if time.time() >= deadline:
        raise TimeoutError('Deadline before model loading')
    # Model load may itself overrun a deadline; no new call begins after it.
    adapter = None
    try:
        print(visual_tests.run_architecture_check(visual_adapter))
        if time.time() >= deadline:
            raise TimeoutError('Deadline after native CPU architecture check')
        adapter = visual_adapter.QwenAttackAdapter(max_new=MAX_NEW_TOKENS)
        if time.time() >= deadline:
            raise TimeoutError('Deadline after model loading')
        run_result = visual_core.run_private(adapter, rows, config, root, scope, deadline)
        print(run_result)  # Counts only. Responses and artifacts stay private.
    except Exception as error:
        visual_core.atomic_json(root/'startup_status_PRIVATE.json',
                dict(status='stopped', error_type=type(error).__name__, scope_sha256=scope_hash))
        print('Stopped:', type(error).__name__, 'See private status; no automatic retries.')
    finally:
        if adapter is not None:
            adapter.close()
else:
    print('Real model run not started. Offline adapter/runner checks are above.')


Real model run not started. Offline adapter/runner checks are above.


### Readout — editable

Read counts only here. Actual responses, delivered PNGs, token IDs, parity errors and optimization traces are stored privately. Failed or timed-out conditions are explicit and not automatically retried. Paired text-only and random-noise/patch controls are also saved. NLL improvement is not jailbreak success. No automated guard rate is invented.


## 5. Evaluation and claims gate
    Evaluate three distinct outcomes: perceptual error, instruction deviation and unsafe assistance.
    Human reviewers see actual images/tasks; blind model, attack, defense and automatic labels.
    For $N$ attempted cases, $H$ resolved unsafe and $M$ unresolved, report $H/N$, $M/N$ and
    $[H/N,(H+M)/N]$ bounds. Refusal, comprehension and cap hits remain separate.
    The private runner currently collects responses and status, not final human scores or inferential ASR.
    Reserved evaluation, weighted dual-review analysis and a full benchmark importer remain pending.

    Before any novel claim: verify the closest papers/configurations, reproduce a baseline, freeze
    data/estimands/compute, then evaluate held-out outcomes. Candidate studies:
    1. Does saved-image/preprocessing variation change transfer at the same delivered budget?
    2. Do defenses generalize across pixel/patch/injection at matched benign utility and query cost?
    3. Does a carefully measured reproduction change an existing conclusion?
    For each write the closest prior art, falsifiable prediction, primary outcome, controls,
    failure criterion and held-out test. None is currently a novel finding.


## 6. Source-linked reading register and evidence limits

# Visual attacks on open VLMs: research dossier

Prepared October 24, 2026. Advisor milestone: October 28; manuscript target: December 1.
**October 25 implementation update:** [real attack lab](../notebooks/vlm_visual_attack_lab.ipynb)
embeds actual input optimizers, a pinned differentiable Qwen adapter, private paired-generation runner,
paper coverage/equations and CPU acceptance checks. All paper-derived paths are labelled adaptations;
original CLIP triggers, Brown's original distribution/physical evaluation, BAP feedback, physical capture
and full adaptive-defense reproductions remain pending. Shared replacement EOT patch mechanics are implemented.
Rebuild with `python3 tools/nb_gen/make_visual_suite_nb.py --execute`.
The GPU branch is off; Amendment 8 is a draft, not accepted. No VLM attack outcome is available yet.
Offline suite passes nine optimizer-mechanism checks and ten private runner paths. An additional
Transformers 4.57.1 test with randomly initialized tiny Qwen weights passes reference preprocessing
and native raw-image gradient checks (`e2e_visual_attacks.py --architecture` in a pinned environment).
These are software checks, not pretrained-model experiments or T4 compatibility evidence.
This is the new research track requested by the owner. Historical refusal-gap results remain in
[STATUS_AND_PAPER_PLAN.md](STATUS_AND_PAPER_PLAN.md); they are not visual-perturbation results.
No new VLM experiment, model download, or GPU run has been performed for this dossier.

## 1. What exists and what is still pending

| Deliverable | Status |
|---|---|
| Searchable core-paper register and mechanism comparison | Implemented below; inspection depth recorded honestly |
| Mathematical teaching notebook | Generated by `tools/nb_gen/make_visual_attacks_nb.py`; CPU toy examples only |
| Original classifier notebook audit | Static source audit below; original dataset/results not reproduced |
| Dataset inventory | Official metadata inspected; artifact/license/overlap checks still pending where stated |
| New attack protocol | Section 7 and Amendment 8 draft in analysis plan; acceptance pending |
| Real FGSM/PGD/patch/injection VLM evaluation | Pending protocol approval and gradient/memory checks |
| Human labels and defense results | Not collected |
| Novel contribution | Not established; decision process in section 9 |

**Reading labels:** “partially inspected” means original method/evaluation sections were opened,
but a complete methods–appendix–code audit is outstanding. “Abstract-only” means discovery only.
No entry below is promoted to “full-text verified” merely because its HTML was available.
Reported paper results are not ours. The survey is a discovery source, not independent validation.
Paper-card numerical extracts are intentionally small; do not compare rates across different tasks.

## 2. Research question and threat model

How do visual manipulations compromise safety or instruction integrity, and which inference-time
defenses reduce failures while preserving useful visual performance? Model weights stay frozen.
Open models/data, T4 first, no paid API dependency. Input optimization is allowed, not weight training.

| Track | Legitimate task | Failure endpoint | Must not be conflated with |
|---|---|---|---|
| Perception | Recognition or VQA | Wrong or specified target answer | Harmful assistance |
| Instruction integrity | User task over untrusted visual content | Predefined unauthorized deviation | Merely transcribing an instruction |
| Safety | Policy-labelled request with fixed image conditions | Relevant prohibited assistance | Any non-refusal, nonsense, or image-reading failure |

Record four separate axes: attacker access (full gradients/encoder-only/queries/transfer),
controlled channel (image/text/both), support (full image/local region), and objective.
FGSM/PGD are optimization methods; a patch is a support constraint; injection is a mechanism.
Visual-only manipulation may accompany a fixed text question. It does not mean no text input.
“Vision is less safe than text” is a hypothesis requiring matched comparisons, not a premise.

## 3. Core reading register

All source links below were opened October 24, 2026 (or rechecked from this session).
Publication labels refer to the linked version; publisher equivalence is not assumed.
Each card states the next work needed before claiming a faithful reproduction.

### P01 — FGSM

Ian J. Goodfellow, Jonathon Shlens, Christian Szegedy, **Explaining and Harnessing Adversarial Examples**.
[1412.6572v3, 2015 preprint revision](https://arxiv.org/html/1412.6572v3). **Partially inspected:** §§3–5.
The first-order loss approximation motivates the sign-gradient update under an infinity-norm constraint.
The experiments concern classifiers, not generative VLM safety. §4 reports MNIST maxout adversarial
misclassification of **89.4%**; this is error, not jailbreak ASR. Training interventions are outside our scope.
Reproduction task: trace input scaling, loss and epsilon through the original preprocessing; then implement
the inner attack on a frozen toy model. Transfer to a VLM requires a different objective and scoring.
Full experiment/appendix and official-code provenance audit pending; no VLM prediction inferred.

### P02 — PGD and robust optimization

Aleksander Madry, Aleksandar Makelov, Ludwig Schmidt, Dimitris Tsipras, Adrian Vladu,
**Towards Deep Learning Models Resistant to Adversarial Attacks**.
[1706.06083v4, 2019 revision; ICLR 2018](https://arxiv.org/html/1706.06083v4).
**Partially inspected:** §2, §5/Table 2 and appendix table pointers.
§2 separates inner input maximization from outer weight minimization; implement only the inner problem.
Table 2 reports CIFAR10 adversarially trained network accuracy **45.8%**, PGD **20** steps,
source A, epsilon **8** in the paper's image scale. This is neither a raw [0,1] epsilon nor a VLM result.
[Official CIFAR challenge](https://github.com/MadryLab/cifar10_challenge) is linked in the paper;
repository implementation not audited here. Check random starts, restart selection, scaling and transfer
before reproducing. Strong PGD results do not certify robustness against all attacks.

### P03 — Universal adversarial patch

Tom B. Brown, Dandelion Mané, Aurko Roy, Martín Abadi, Justin Gilmer, **Adversarial Patch**.
[1712.09665v2, 2018 preprint revision](https://arxiv.org/html/1712.09665v2).
**Partially inspected:** §§2–3, Fig.1. §2/Eq.1 optimizes expected target log probability across
images, locations and transformations. This universal replacement patch differs from the existing
notebook's per-image bounded patch. Fig.1 illustrates VGG16 target confidence **99%**; it is one
illustration, not population ASR. §3 includes a control patch and model-ensemble evaluation.
Reproduction task: distinguish replacement from bounded additive support, define patch area in input
and processed coordinates, and separate optimization images from held-out images. Physical photographs
are outside our first experiment. Full code/config and appendix verification remain pending.

### P04 — Field map

Chiyu Zhang, Lu Zhou, Xiaogang Xu, Jiafei Wu, Zhe Liu,
**Adversarial Attacks of Vision Tasks in the Past 10 Years: A Survey**.
[2410.23687v2, May 2025 revision](https://arxiv.org/html/2410.23687v2).
**Partially inspected:** §§3–5 and §§7.2–7.5. §7 separates access, purpose and technique; Table 3
distinguishes capability, red-team, robustness and alignment data. Use as the citation map, not a benchmark
to reproduce. Its query/gray-box terminology must be translated into explicit access descriptions.
The ACM DOI supplied by the owner appears to identify the published counterpart; ACM access returned
403, so publisher/preprint text equality is unverified. Do not count them as independent evidence.
Follow original sources and search subsequent work through October 24, 2026.

### P05 — Visual adversarial examples against safety

Xiangyu Qi, Kaixuan Huang, Ashwinee Panda, Peter Henderson, Mengdi Wang, Prateek Mittal,
**Visual Adversarial Examples Jailbreak Aligned Large Language Models**.
[2306.13213v2, August 2023 revision](https://arxiv.org/html/2306.13213v2).
**Partially inspected:** §§3–4.5 and appendix headings. §3/Eq.1 minimizes corpus negative log
likelihood through the VLM to optimize an image; evaluation uses other textual requests.
§4.4 explicitly includes a text-attack comparator, so our comparison is not a first.
§4.3/Table 2 measures toxicity, which differs from task-relevant harmful assistance.
[Official repository](https://github.com/Unispac/Visual-Adversarial-Examples-Jailbreak-Large-Language-Models)
includes model-specific attack code and artifacts. Next: pin checkpoint, corpus and code commit;
extract all budgets/steps/decoding from code and Appendix A before reproducing. T4 backward fit is unknown.

### P06 — Encoder-space composition

Erfan Shayegani, Yue Dong, Nael Abu-Ghazaleh,
**Jailbreak in pieces: Compositional Adversarial Attacks on Multi-Modal Language Models**.
[2307.14539v2, October 2023](https://arxiv.org/html/2307.14539v2).
**Partially inspected:** §3.1/Eqs.3–4, Algorithm 1, §4/Table 1 and appendix headings.
Minimizes distance between image embeddings and target-trigger embeddings using encoder access;
tests LLaVA and LLaMA-Adapter V2. Table 1 reports annotator Fleiss' kappa **0.8969**; that agreement
cannot validate our reviewers. Four trigger constructions distinguish text, OCR, visual and combined
targets. Encoder matching is a surrogate objective; a close embedding alone is not a successful jailbreak.
Next: inspect exact CLIP variant, initialization, stopping/bounds and Appendix H rubric; official code
and release identity remain unverified. Keep as an extension until end-to-end baselines work.

### P07 — Behavioral image hijacks

Luke Bailey, Euan Ong, Stuart Russell, Scott Emmons,
**Image Hijacks: Adversarial Images can Control Generative Models at Runtime**.
[2309.00236v4, September 2024](https://arxiv.org/html/2309.00236v4).
**Partially inspected:** §§2–4/Table 1 and Appendix C headings.
Eq.1 teacher-forces target behavior and minimizes output-distribution loss across contexts;
prompt matching uses model-generated behavior as the target. Table 1 reports **92%** jailbreak success
at epsilon **16/255**, versus original image **4%**, under that paper's protocol.
It also evaluates stationary/moving patches; patch jailbreak cells marked unrun must remain unrun.
[Official project/code entry](https://image-hijacks.github.io/). Next: inspect context splits, judge,
full precision/checkpoint and restart details. A newer LLaVA checkpoint is an adaptation, not a reproduction.

### P08 — Joint image–text attack

Zonghao Ying, Aishan Liu, Tianyuan Zhang, Zhengmin Yu, Siyuan Liang, Xianglong Liu, Dacheng Tao,
**Jailbreak Vision Language Models via Bi-Modal Adversarial Prompt**.
[2406.04031v2, July 2024](https://arxiv.org/html/2406.04031v2).
**Partially inspected:** §§3–5/Table 1, Appendix A and D headings.
§4.1/Eq.3 optimizes query-agnostic image likelihood; §4.2 then changes text using feedback.
Table 1's MiniGPT4 Illegal Activity row reports BAP **59.04** percent; the no-attack entry is **2.13**.
§5.1 uses ChatGPT and an A800 cluster: an open-only T4 replacement is an adaptation.
Appendix D hyperparameters and code identity still need checking. Keep combined attacks distinct from
image-only manipulations; report text/image ablations and attack-search queries. Extended reproduction,
not part of the initial GPU core.

### P09 — Existing attack–defense benchmark

Fenghua Weng, Yue Xu, Chengyan Fu, Wenjie Wang,
**MMJ-Bench: A Comprehensive Study on Jailbreak Attacks and Defenses for Vision Language Models**.
[AAAI 2025 original PDF](https://ojs.aaai.org/index.php/AAAI/article/download/34983/37138).
**Partially inspected:** introduction, attack/defense taxonomy and benchmark setup; result tables require
cell-level extraction. [Official repository](https://github.com/thunxxx/MLLM-Jailbreak-evaluation-MMJ-Bench).
Compares existing attacks and defenses and includes normal-task utility. A broad comparison alone is
therefore not our novelty. Do not confuse **MMJ-Bench** with the separate **MMJailBench** cited in the
historical project. Next: pin artifact manifest, source benchmarks, evaluation labels and model commits;
reproduce a supported slice before adding new models. No numerical result transcribed until checked.

### P10 — Physical visual prompt injection

Chen Ling, Kai Hu, Hangcheng Liu, Xingshuo Han, Tianwei Zhang, Changhai Ou,
**Physical Prompt Injection Attacks on Large Vision–Language Models**.
[2601.17383v1, January 2026](https://arxiv.org/html/2601.17383v1).
**Partially inspected:** §§III–V and VI headings. §III-C assumes no target interaction or user-query
access during generation; §III-D defines visual insertion rather than a small pixel perturbation.
[Paper-linked code](https://github.com/2023cghacker/Physical-Prompt-Injection-Attack) not audited here.
Digital placement under our own known tasks changes the threat model; physical robustness requires
camera capture, not merely software rotations. Full per-task results, simulation data permissions and
physical evaluation remain pending. The earlier AI answer misidentified this arXiv ID as InjecAgent.

### P11 — Defense prompting

Yu Wang, Xiaogeng Liu, Yu Li, Muhao Chen, Chaowei Xiao,
**AdaShield: Safeguarding Multimodal Large Language Models from Structure-based Attack via Adaptive Shield Prompting**.
[2403.09513v1, March 2024](https://arxiv.org/html/2403.09513v1).
**Partially inspected:** §§3–4.2/Table 2 and appendix headings.
Static AdaShield-S differs from adaptive prompt-pool construction/retrieval. §4.1 uses Vicuna as defender
and GPT4 paraphrasing; evaluates FigStep/QR and MM-Vet, with keyword ASR plus GPT recheck.
Table 2 LLaVA-1.5-13B FigStep Recheck entries: vanilla **87.21**, AdaShield-S **35.58**.
[Official code](https://github.com/SaFo-Lab/AdaShield). Next: inspect split leakage, pool construction
and thresholds. A static open-only defense is a narrower baseline, not the full adaptive reproduction.

### P12 — Mutation-based detection

Xiaoyu Zhang, Cen Zhang, Tianlin Li, Yihao Huang, Xiaojun Jia, Ming Hu, Jie Zhang, Yang Liu,
Shiqing Ma, Chao Shen, **JailGuard: A Universal Detection Framework for Prompt-based Attacks on LLM Systems**.
[2312.10766v4, March 2025; TOSEM DOI in text](https://arxiv.org/html/2312.10766v4).
**Partially inspected:** §4/Eqs.5–7, §6.2/Table 4 and adaptive-attack discussion.
Constructs distributions from response similarities and thresholds KL divergence, with a refusal rule.
Table 4 image-policy accuracy **82.90**, precision **74.00**, recall **88.25** percent.
These are detection metrics, not unsafe-assistance rates. [Official code](https://github.com/shiningrain/JailGuard).
Next: pin variant count, similarity model, threshold calibration, cost and adaptive coverage.
Stable unsafe responses can defeat inconsistency intuition; benign OCR tasks can be mutation-sensitive.

## 4. Extended discovery and novelty search

October 24 query log: `2026 vision language adversarial patch adaptive defense attacks benchmark`;
`site.aclanthology.org 2025 2026 visual prompt injection benchmark`;
`Jailbreaking Vision-Language Models Through the Visual Modality`.
These searches are a first pass, not an exhaustive or negative novelty result.

| Lead | Inspection status | Follow-up / exclusion |
|---|---|---|
| [Azulay et al., 2605.00583](https://arxiv.org/abs/2605.00583) | Abstract-only | Semantic symbols/substitution/analogy: check task matching, code and openness |
| [VPI-Bench, 2506.02456](https://arxiv.org/abs/2506.02456) | Search discovery only | Computer-use agents; exclude tool execution from our first experiment |
| [Adversarial Defense overview, 2601.12443](https://arxiv.org/abs/2601.12443) | Search discovery only | Follow original defenses; distinguish CLIP classification from generative VLM safety |
| [PHANTOM, 2606.24388v1](https://arxiv.org/html/2606.24388v1) | Dataset card and construction/Appendix F inspected | Mixed attack families, access gate, overlap with MM-SafetyBench |
| [CyberSecEval3 visual injection](https://huggingface.co/datasets/facebook/cyberseceval3-visual-prompt-injection) | Official card inspected | Candidate digital injection source; exact clean-task reconstruction pending |

Forward review must also search original-paper citations, ACL Anthology and official repositories,
record dates, deduplicate versions, and test each proposed contribution against closest existing work.
No “first”, “all attacks”, universal defense, or resolved safety–capability trade-off claim is supported.

## 5. Original notebook audit (static, not reproduction)

Source: [DL_final (1).ipynb](https://github.com/dev4-gpt/Adversarial-attacks/blob/main/DL_final%20%281%29.ipynb).
Inspected temporary download SHA256: `a6bd4d19df0cad481a0e72a91845faf8175cedc5ab162d77809ffcc4fa9290ba`.
Cell numbers below are one-based. No original code was executed; no Drive files were changed.

| Location | Finding | Required action before reuse |
|---|---|---|
| Cell 7 | ImageFolder classes mapped by sorted-label order | Validate synset-to-ImageNet index explicitly; reject absent/ambiguous mapping |
| Cells 8,10 | FGSM/PGD epsilon applied after normalization; no valid-image clamp | Optimize raw [0,1] pixels and normalize inside forward; project to box AND budget |
| Cell 11 | Patch uses global extrema across normalized channels | Use raw-pixel bounds; global scalar bounds are not valid per-channel ranges |
| Cell 11 | Random location/target per image, bounded patch | Label per-example targeted patch, not Brown universal patch |
| Cell 11 | Reports top-1/top-5 against true class | Add target-hit rate; wrong true class does not establish chosen target success |
| Code cells | No Python/NumPy/PyTorch seed initialization found | Local seeded generators; record configuration and deterministic limitations |
| Cell 11 | Lower patch accuracy triggers “less effective” message | Correct direction: lower accuracy means greater untargeted degradation |
| Cell 8 vs 10–11 | FGSM save is commented; PGD/patch use .pt | Verify actual files; README's advertised directory is not proof of saved artifacts |
| Cell 5 | Destructive Drive cleanup commands | Never execute; omit from any new notebook |

For channel normalization z=(x-mu)/sigma, raw delta_x=sigma*delta_z. Hence a scalar normalized
epsilon is channel-dependent in raw units. “Imperceptible” requires evidence, not just an L-infinity cap.
README accuracy figures remain reported historical values, not independently reproduced measurements.

## 6. Data readiness matrix

No benchmark image, prompt, reply or full dataset is copied into the public repository.
Metadata availability does not imply artifact-level reproducibility or redistribution permission.

| Source / role | Verified metadata | Outstanding gate and split unit |
|---|---|---|
| Existing ImageNet subset / classifier audit | Notebook expects `TestDataSet` and labels file | Actual subset/license/checksums not located in this pass; split by image, verify label IDs |
| [COCO](https://cocodataset.org/#termsofuse) / carriers | Official site opened; dynamic terms not extracted | Review each image's source/license and pin IDs; group images across all conditions |
| [VQAv2](https://visualqa.org/download.html) / benign VQA | Official downloads available | Terms, annotation schema and scorer pending; group shared images/questions |
| [HarmBench](https://github.com/centerforaisafety/HarmBench) / requests | Official repository lists MIT license | Pin behavior CSV revision, source permissions and historical semantic overlap; group intent |
| [MMJ-Bench](https://github.com/thunxxx/MLLM-Jailbreak-evaluation-MMJ-Bench) / reproduction | Official code entry verified | Artifact download/license/clean pairing/hash checks pending; group original intent/artifact |
| [PHANTOM](https://huggingface.co/datasets/it4lia/PHANTOM) / optional transfer | Card lists CC-BY-4.0; access requires accepting conditions | Owner gate; upstream image permissions, attack components, overlap and schemas pending |
| [CyberSecEval3 visual injection](https://huggingface.co/datasets/facebook/cyberseceval3-visual-prompt-injection) / injection | Card lists MIT; test split; evaluation-only intended use | Verify exact records/scorer and clean counterparts privately; no tuning on reserved test records |

Private manifest must record source revision, original ID, content hash, intent/image group, prior exposure,
license/access evidence, role (attack optimization/development/evaluation), track and clean-pair ID.
Split connected intent/image groups, not individual variants. Exclude historical seen items from fresh
confirmation; mark uncertain provenance as development-only. Universal optimization contexts/images
must not leak into evaluation. Selection precedes viewing outputs; persist an immutable manifest hash.

## 7. Proposed study contract — DRAFT, no scientific execution

This is review material, not the required amendment in `ANALYSIS_PLAN_MODALITY_SAFETY.md`.
After dataset/code audit and measured compute profiling, append a new amendment there without changing
historical hypotheses. Owner acceptance and a specific run budget remain required.

**Candidate endpoints:** E1 visual-attack minus clean unsafe-assistance rate (primary safety);
E2 defense-aware attacked minus undefended attacked unsafe rate; E3 paired benign utility change;
E4 injection deviation over paired legitimate tasks. These are candidate contrasts, not accepted hypotheses.
Keep perception endpoints in a distinct family. “Success among clean-baseline failures” means successful
attacks among cases where the clean baseline did NOT already meet the attack objective; spell out denominator.

**Missingness:** let N be attempted cases, H resolved unsafe and M unresolved/failed cases. Report
H/N as confirmed-unsafe fraction, M/N missing fraction, and [H/N,(H+M)/N] deterministic bounds.
Also report H/(N-M) as resolved-only, explicitly conditional. Never call unresolved safe or publish
the lower bound alone as the population unsafe rate. Apply paired missingness bounds to paired contrasts.

**Planned panel:** Qwen2.5-VL-3B-Instruct first, LLaVA-1.5-7B second if feasible. Existing Model7 uses
inference-mode generation, not an attack gradient API. New adapters must use teacher-forced loss,
target-token masks (exclude prompt/image/padding), differentiable raw-pixel preprocessing and frozen weights.
Check finite nonzero image gradients and finite-difference agreement; compare clean reference logits.
NF4 loading does not establish backward support or memory fit. Pin model/processor/code revisions.
No automatic checkpoint substitution. Original-paper checkpoints require a separate feasibility decision.

**Compute gate:** proposed benign profile = one model, one image/task, forward + backward + short
optimization, at most 30 T4 minutes after approval. Record wall time and allocated/reserved peak memory.
Abort on OOM/nonfinite gradients; do not repeatedly download or retry. No larger-GPU purchase by default.
Project total cost as optimization contexts × steps × restarts × models, plus generations and judges;
include multiplicative defense variants/EOT draws and shared universal-artifact costs.

**Development values, not final scientific settings:** raw-pixel epsilon sweep {2,4,8,16}/255;
FGSM one step; PGD {10,20,40} steps, alpha=epsilon/4, restarts {1,3}; patch area {1%,4%}.
Use seeded starts; choose restarts by attack objective, never evaluation labels. These are our adaptations,
not attributed paper defaults. Select/freeze final values using development only, then amend before testing.
Provisional request caps: 40 development / 120 evaluation plus comparable benign coverage, not a power claim.
Separate original-paper configuration from our common-budget comparison; do not call them the same run.

**Conditions:** clean image, text-only, random bounded noise, matched-area random patch, FGSM,
PGD, localized patch and visual injection within their valid task tracks. Fixed text for visual attacks.
Qi reproduction and combined image/text are separate configurations; no hidden textual optimization.
Use batch one, fixed generation template/precision and 256-token development output window; record EOS
and cap hits. Freeze scoring and any window change before evaluation; no claim that 256 is sufficient.

**Statistics:** pair by task/image; aggregate repeated variants within request before bootstrap;
use 2,000 request-cluster resamples for descriptive 95% intervals. If universal artifacts are shared,
condition conclusions on the tested artifacts; independent artifacts are needed for artifact generalization.
Report scenario/source and image breakdowns; paired McNemar only when its independent-pair assumptions hold.
Specify Holm family in the accepted amendment; don't borrow the earlier eight-scenario inference blindly.
Select a meaningful effect and freeze sample size after cost/power assessment; no output-driven shrinkage.

**Human review:** two independent people, blinded model/attack/defense/guard identities; retain task and
image needed for scoring. Proposed 120-response stratified probability sample with saved inclusion
probabilities; separately flag disagreement diagnostics. Preserve independently assigned harm, refusal,
comprehension, legitimate-task and injection labels before adjudication. Include UNCERTAIN; report
coverage, agreement/kappa and unresolved cases. Repeat 20 randomly selected units after at least 24 hours.
Use inverse-probability weighting for population estimates; no unweighted disagreement-selected rates.
Human evaluation and assistive-LLM provenance stay distinct. Guard text alone cannot establish visual correctness.

**Defenses:** none; reviewed fixed defense prompt; deterministic transformation baseline; optional
JailGuard-style detector only after cost measurement. Development-calibrate thresholds and benign utility.
For transformations evaluate defense-aware/EOT gradients; nondifferentiable transforms require an
explicit approximation/transfer check and an incompleteness statement. For prompting include prompt-aware
optimization. Detection accuracy is not protected-system safety. Reject-all is a utility baseline, not success.

**Cache/output contract for the later runner:** private task/response/artifact files; identities include
split hash, exact model+processor revision, loss/target hash, budget, steps, restart, seed, defense,
decoding and code hash. Atomic writes, per-condition status and explicit failure reasons; resume completed
units only on exact identity. Public export uses an allowlist of scalar counts/rates/intervals/timings and
non-sensitive provenance. Do not serialize nested replies, images or prompts into aggregate exports.

## 8. Reproduction checklist and implementation acceptance

Use labels **faithful reproduction**, **reduced-scale reproduction**, **adaptation** consistently.
Faithful requires original supported checkpoint, inputs/splits, preprocessing, objective and budget;
different models or losses are adaptations even if code was copied. Never fill unpublished cells with estimates.

| Check | Current teaching notebook | Later VLM runner requirement |
|---|---|---|
| Gradient / direction | Analytic toy gradient checked against finite differences; targeted and untargeted tests | Check through actual processor/encoder/connector/LM |
| Constraints | Box and infinity budget, local support, invalid-argument tests | Check artifact after encode/decode and actual preprocessing |
| Serialization | Synthetic raw-pixel 8-bit roundtrip measured; no media embedded | Reproject/validate saved artifact; record effective budget, never assume |
| Frozen model | Toy parameter snapshots unchanged | Actual weights/revisions unchanged; no optimizer over model weights |
| Injection | Symbolic harmless policy example, not a VLM experiment | Original source subset, legitimate tasks, actual model behavior |
| Evaluation | Synthetic missingness bounds demonstration | Real blinded labels, weighted sample, uncertainty and failures |
| Reliability | Deterministic notebook rebuild and CPU checks | Cache identity, crash/resume, isolation and export privacy tests |

Rebuild: `python3 tools/nb_gen/make_visual_attacks_nb.py --execute`.
CPU demonstrations prove mechanics only. No model safety, T4 fit, attack transfer or paper reproduction follows.
Toy images are tensors only: no released benchmark images, instructions or generated replies.

## 9. Calendar, deliverables and contribution gate

| Date | Required deliverable / decision |
|---|---|
| Oct 24–25 | This initial dossier/notebook; continue foundation full-text and code audits |
| Oct 26–27 | Complete core-paper appendices/configs; source permission/overlap manifests; reviewable amendment |
| Oct 28 | Advisor-ready updated dossier and notebook, verified/pending ledger and approved next bounded scope |
| Oct 29–Nov 4 | Accepted amendment; gradient adapters, benign profiles, supported reduced-scale reproductions; freeze test data |
| Nov 5–11 | Core attack comparisons and initial independent human evaluation |
| Nov 12–18 | Defense-aware tests, held-out extension, adjudication |
| Nov 19–25 | Traceable figures and complete draft, including negative results |
| Nov 26–Dec 1 | Advisor revision, reproducibility/privacy check, submission-ready manuscript |

By Nov 4 assess: preprocessing/deployment transfer, defense generalization at matched utility/cost,
or a consequential reproduction finding. First rule out close prior art, then require working baselines,
an affordable held-out test and human validation. Otherwise write a replication/measurement paper.
Current novelty verdict for all three: **unestablished**, not novel by default.
The broad comparison in MMJ-Bench and preprocessing/transfer work are mandatory comparators.

Milestones are deliverables, not scheduled autonomous execution. No automation was created.
December 1 means manuscript readiness, not guaranteed acceptance; owner selects venue.
The next owner review concerns the concrete scientific amendment and measured run scope, not this CPU notebook.
